# Bluesky Trending Topics (Python Notebook)

This notebook reimplements the Bluesky trending topic extractor entirely in Python. It mirrors the TypeScript pipeline by preprocessing text, filtering stopwords/blacklisted terms, extracting n-grams (words, short phrases, hashtags), and ranking trends within a rolling time window.


## Environment setup

This notebook only depends on the Python standard library. To run it:

1. (Optional) create a virtual environment.
2. Install Jupyter with `pip install notebook`.
3. Launch `jupyter notebook` and open `bsky_trends.ipynb`.

> Tip: The stopword and blacklist vocabularies are loaded from the repository's existing TypeScript assets so you get the same filtering behavior as the original project.


In [ ]:
from __future__ import annotations
import ast
import math
import re
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from typing import Iterable


In [ ]:
TS_SET_PATTERN = re.compile(r"new Set\(\[(.*)\]\)", re.DOTALL)


def load_ts_set(path: str) -> set[str]:
    """Parse a TypeScript `new Set([...])` file into a Python set."""
    content = Path(path).read_text(encoding="utf-8")
    match = TS_SET_PATTERN.search(content)
    if not match:
        raise ValueError(f"Could not parse set from {path}")
    payload = match.group(1)
    values = ast.literal_eval(f"[{payload}]")
    return set(str(v).lower() for v in values)


stopwords_en = load_ts_set("src/filters/stopwords/stopwords_en.ts")
stopwords_pt = load_ts_set("src/filters/stopwords/stopwords_pt.ts")
blacklist = load_ts_set("src/filters/blacklist.ts")


In [ ]:
URL_PATTERN = re.compile(r"https?://[^\s]+", re.IGNORECASE)
DOMAIN_PATTERN = re.compile(r"(?:[a-z0-9-]+\.)+[a-z]{2,6}(?:/[^\s]*)?", re.IGNORECASE)
SLASH_COMMAND_PATTERN = re.compile(r"\[a-zA-Z0-9]+")
PUNCTUATION_PATTERN = re.compile(r"[@*%$(){}<>[\],.;:"'^&|~`]")


def preprocess_text(text: str) -> str:
    """Mirror the original cleaning rules while keeping hashtags and ?/! punctuation."""
    cleaned = URL_PATTERN.sub("", text)
    cleaned = DOMAIN_PATTERN.sub("", cleaned)
    cleaned = SLASH_COMMAND_PATTERN.sub("", cleaned)
    cleaned = PUNCTUATION_PATTERN.sub("", cleaned)
    cleaned = cleaned.lower().strip()
    cleaned = cleaned.encode("ascii", "ignore").decode("ascii")
    return cleaned


def extract_hashtags(text: str) -> list[str]:
    return re.findall(r"#[a-z0-9]+", text, flags=re.IGNORECASE)


def extract_sentences(text: str) -> list[str]:
    candidates = re.split(r"[.!?]+", text)
    return [c.strip() for c in candidates if c.strip()]


def extract_words(text: str) -> list[str]:
    tokens = re.findall(r"#?[^\W_]+", text, flags=re.UNICODE)
    return [t for t in tokens if not t.startswith("#")]


stopwords_unified = stopwords_en | stopwords_pt
blacklist_patterns = [
    re.compile("^" + re.escape(token).replace(r"\*", ".*").replace(r"\+", ".+") + "$", re.IGNORECASE)
    for token in blacklist
]


def filter_sentences(sentences: Iterable[str]) -> list[str]:
    filtered = []
    for sentence in sentences:
        words = sentence.split()
        if len(words) < 2 or len(words) > 3:
            continue
        if len(set(words)) != len(words):
            continue
        if any(word.startswith("#") for word in words):
            continue
        lower_words = [w.lower() for w in words]
        if any(pattern.search(w) for w in lower_words for pattern in blacklist_patterns):
            continue
        filtered.append(sentence)
    return filtered


def filter_words(words: Iterable[str], _lang: str) -> list[str]:
    results: list[str] = []
    for raw in words:
        word = raw.lower().lstrip("#")
        if len(word) <= 1 or " " in word:
            continue
        if any(p.search(word) for p in blacklist_patterns):
            continue
        if word in stopwords_unified and not raw.startswith("#"):
            continue
        results.append(raw)
    return results


In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Iterable


@dataclass
class TrendEntry:
    term: str
    timestamps: list[datetime]

    @property
    def last_seen(self) -> datetime:
        return max(self.timestamps)

    @property
    def count(self) -> int:
        return len(self.timestamps)


class TrendTracker:
    def __init__(self, max_age_hours: int = 6, decay: float = 0.97):
        self.max_age = timedelta(hours=max_age_hours)
        self.decay = decay
        self.events: dict[str, dict[str, TrendEntry]] = {
            "words": {},
            "phrases": {},
            "hashtags": {},
        }

    def _touch(self, kind: str, term: str, timestamp: datetime) -> None:
        bucket = self.events[kind]
        entry = bucket.get(term)
        if entry is None:
            bucket[term] = TrendEntry(term=term, timestamps=[timestamp])
        else:
            entry.timestamps.append(timestamp)

    def ingest(self, *, words: Iterable[str], phrases: Iterable[str], hashtags: Iterable[str], timestamp: datetime) -> None:
        for term in words:
            self._touch("words", term, timestamp)
        for term in phrases:
            self._touch("phrases", term, timestamp)
        for term in hashtags:
            self._touch("hashtags", term, timestamp)

    def _prune(self, now: datetime) -> None:
        cutoff = now - self.max_age
        for bucket in self.events.values():
            expired = []
            for term, entry in bucket.items():
                entry.timestamps = [ts for ts in entry.timestamps if ts >= cutoff]
                if not entry.timestamps:
                    expired.append(term)
            for term in expired:
                del bucket[term]

    def _score(self, entry: TrendEntry, now: datetime) -> float:
        hours_since_last = (now - entry.last_seen).total_seconds() / 3600
        return entry.count * (self.decay ** hours_since_last)

    def snapshot(self, *, limit: int = 10, min_count: int = 5) -> dict[str, list[dict[str, object]]]:
        now = datetime.utcnow()
        self._prune(now)
        results: dict[str, list[dict[str, object]]] = {}
        for kind, bucket in self.events.items():
            scored = [
                {
                    "term": entry.term,
                    "count": entry.count,
                    "last_seen": entry.last_seen.isoformat(),
                    "score": self._score(entry, now),
                }
                for entry in bucket.values()
                if entry.count >= min_count
            ]
            scored.sort(key=lambda item: item["score"], reverse=True)
            results[kind] = scored[:limit]
        return results


In [ ]:
def process_post(post: dict, tracker: TrendTracker) -> None:
    text = preprocess_text(post["text"])
    lang = post.get("lang", "en")
    timestamp = datetime.fromisoformat(post.get("created_at", datetime.utcnow().isoformat()))

    sentences = filter_sentences(extract_sentences(text))
    words = filter_words(extract_words(text), lang)
    hashtags = filter_words(extract_hashtags(text), lang)

    tracker.ingest(words=words, phrases=sentences, hashtags=hashtags, timestamp=timestamp)


## Demo: processing a small batch of Bluesky-style posts

The sample below mixes Portuguese and English content to demonstrate language-aware stopword filtering and basic trend scoring. Replace the `sample_posts` list with data from your own feed to experiment.


In [ ]:
sample_posts = [
    {"text": "🇧🇷 Vamos falar sobre #tecnologia e inovação no Brasil!", "lang": "pt", "created_at": "2024-05-01T12:00:00"},
    {"text": "Incrível como a comunidade de #opensource cresce todo dia.", "lang": "pt", "created_at": "2024-05-01T12:05:00"},
    {"text": "What a game last night! Amazing defense and teamwork.", "lang": "en", "created_at": "2024-05-01T12:10:00"},
    {"text": "The new AI update is wild, can't believe the speed! #ai", "lang": "en", "created_at": "2024-05-01T12:15:00"},
    {"text": "AI está transformando tudo, inclusive o #futebol com estatísticas avançadas.", "lang": "pt", "created_at": "2024-05-01T12:20:00"},
    {"text": "opensource communities keep sharing great #ai notebooks.", "lang": "en", "created_at": "2024-05-01T12:25:00"},
    {"text": "Que defesa sensacional no jogo de ontem, pura garra!", "lang": "pt", "created_at": "2024-05-01T12:30:00"},
    {"text": "AI notebooks make rapid prototyping a breeze. #ai #opensource", "lang": "en", "created_at": "2024-05-01T12:35:00"},
]

tracker = TrendTracker(max_age_hours=6)
for post in sample_posts:
    process_post(post, tracker)

snapshot = tracker.snapshot(limit=5, min_count=1)
snapshot


## Interpreting the results

The `snapshot` object mirrors the REST endpoint from the original service. Each entry includes the raw term, how many times it appeared in the recent window, when it was last seen, and a decay-adjusted score you can use to rank trends.

To adapt this notebook to a production setting:

- Swap `sample_posts` for a stream of posts from the Bluesky API.
- Persist `tracker.events` to disk or a database between runs.
- Expose `snapshot()` through your preferred web framework (e.g., FastAPI or Flask) if you need an HTTP endpoint.
